## XGBoost Regreesion and Classification

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("train.csv")
df.head()

,Hour,Temperature,Humidity(%),Wind speed (m/s),Visibility (10m),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Month_Num,Date_Day,is_weekend,Seasons_Spring,Seasons_Summer,Seasons_Winter,Holiday_No Holiday,Rented Bike Count
0,0.833333,0.639594,0.59375,-0.227684,0.166197,-0.010753,0.0,0.0,0.333333,0.333333,0.0,0.0,1.0,0.0,0.0,1828
1,-0.166667,-0.502538,-0.62500,-0.148918,0.291080,0.881720,0.0,0.0,-0.500000,0.333333,0.0,1.0,0.0,0.0,0.0,374
2,0.833333,0.162437,-0.90625,-0.395753,0.287324,-0.010753,0.0,0.0,-0.333333,0.333333,1.0,1.0,0.0,0.0,0.0,1292
3,-0.166667,-0.304569,-0.12500,-0.581656,-0.733333,1.064516,0.0,0.0,0.833333,0.200000,0.0,0.0,0.0,0.0,0.0,651
4,0.583333,0.477157,-0.68750,0.205842,-0.916432,0.225806,0.0,0.0,-0.333333,0.266667,0.0,1.0,0.0,0.0,0.0,1578


In [3]:
X = df.drop("Rented Bike Count", axis=1)
y = df["Rented Bike Count"]

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
from xgboost import XGBRegressor

xgbr = XGBRegressor()

xgbr.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_met

In [6]:
y_pred = xgbr.predict(X_test)

In [8]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

print("MSE: ", mean_squared_error(y_test, y_pred))
print("MAE: ", mean_absolute_error(y_test, y_pred))
print("RMSE: ", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2: ", r2_score(y_test, y_pred))

MSE:  26371.203125
MAE:  100.64115142822266
RMSE:  162.39212765710042
R2:  0.9375112056732178


## Hyper-Parameter Tunning using Grid Search CV

In [17]:
from sklearn.model_selection import RandomizedSearchCV

# define model
xgbr = XGBRegressor(
    objective="reg:squarederror",
    random_state=42
)

# Parameter distributions
param_dist = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'n_estimators': [100, 200, 300, 500, 700, 1000],
    'max_depth': [3, 4, 5, 6, 8, 10],
    'min_child_weight': [1, 3, 5, 7, 10],
    'gamma': [0, 0.1, 0.3, 0.5, 1, 5],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.01, 0.1, 1, 10],
    'reg_lambda': [0.1, 1, 2, 5, 10]
}

# grid Search
random_search = RandomizedSearchCV(
    estimator=xgbr,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=2
)

# Train
random_search.fit(X_train, y_train)

# Results
print("Best Parameters:")
print(random_search.best_params_)

print("\nBest CV Score:")
print(random_search.best_score_)

# Best Model
best_model = random_search.best_estimator_

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best Parameters:
{'subsample': 0.9, 'reg_lambda': 10, 'reg_alpha': 0.01, 'n_estimators': 700, 'min_child_weight': 1, 'max_depth': 8, 'learning_rate': 0.1, 'gamma': 1, 'colsample_bytree': 1.0}

Best CV Score:
0.9456055164337158


In [18]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

# Predictions
y_pred = best_model.predict(X_test)

# Metrics
print("R² :", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

R² : 0.9505937099456787
MAE: 85.85618591308594
RMSE: 144.3959129875219


## HyperParameter Tunning using BayesSearch CV

In [11]:
from skopt import BayesSearchCV
from skopt.space import Real, Integer

In [14]:
# Model
xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42
)

# Search Space
search_spaces = {
    'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    'n_estimators': Integer(100, 1000),
    'max_depth': Integer(3, 10),
    'min_child_weight': Integer(1, 10),
    'gamma': Real(0.0, 5.0),
    'subsample': Real(0.6, 1.0),
    'colsample_bytree': Real(0.6, 1.0),
    'reg_alpha': Real(1e-5, 10, prior='log-uniform'),
    'reg_lambda': Real(0.1, 10, prior='log-uniform')
}

# Bayesian Optimization
bayes_search = BayesSearchCV(
    estimator=xgb,
    search_spaces=search_spaces,
    n_iter=50,             # Number of parameter sets to evaluate
    scoring='r2',
    cv=5,
    n_jobs=-1,
    verbose=2,
    random_state=42
)

bayes_search.fit(X_train, y_train)

print("Best Parameters:")
print(bayes_search.best_params_)

print("\nBest CV Score:")
print(bayes_search.best_score_)

best_model = bayes_search.best_estimator_

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fi

In [15]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

# Predictions
y_pred = best_model.predict(X_test)

# Metrics
print("R² :", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

R² : 0.9460291266441345
MAE: 89.76519775390625
RMSE: 150.91890413397522


In [16]:
from sklearn.model_selection import GridSearchCV

# Define the model
xgbr = XGBRegressor(
    objective='reg:squarederror',
    random_state=42
)

# Parameter Grid
param_grid = {
    'learning_rate': [0.07, 0.08, 0.09],
    'max_depth': [8, 9],
    'min_child_weight': [4, 6],
    'gamma': [0.8, 1.0],
    'subsample': [0.9, 1.0],
    'colsample_bytree': [0.9, 1.0],
    'reg_alpha': [0.1, 0.3],
    'reg_lambda': [5, 6],
    'n_estimators': [150, 200, 250]
}

# Grid Search
grid_search = GridSearchCV(
    estimator=xgbr,
    param_grid=param_grid,
    scoring='r2',          # or 'neg_mean_squared_error'
    cv=5,
    n_jobs=-1,
    verbose=2
)

# Train
grid_search.fit(X_train, y_train)

# Best Parameters
print("=" * 60)
print("Best Parameters:")
print(grid_search.best_params_)

# Best CV Score
print("\nBest Cross Validation R² Score:")
print(grid_search.best_score_)

# Best Model
best_model = grid_search.best_estimator_

# Evaluate on Test Set
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

y_pred = best_model.predict(X_test)

print("\n" + "=" * 60)
print("Test Set Performance")
print("=" * 60)

print(f"R² Score : {r2_score(y_test, y_pred):.4f}")
print(f"MAE      : {mean_absolute_error(y_test, y_pred):.4f}")
print(f"RMSE     : {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

Fitting 5 folds for each of 1152 candidates, totalling 5760 fits
Best Parameters:
{'colsample_bytree': 0.9, 'gamma': 1.0, 'learning_rate': 0.07, 'max_depth': 9, 'min_child_weight': 4, 'n_estimators': 250, 'reg_alpha': 0.3, 'reg_lambda': 6, 'subsample': 0.9}

Best Cross Validation R² Score:
0.9462528228759766

Test Set Performance
R² Score : 0.9473
MAE      : 87.8854
RMSE     : 149.0920
